In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip uninstall -y py-feat
!pip install py-feat==0.3.5

Found existing installation: py-feat 0.3.5
Uninstalling py-feat-0.3.5:
  Successfully uninstalled py-feat-0.3.5
  Using cached py_feat-0.3.5-py2.py3-none-any.whl


In [2]:
!pip install scipy==1.10.1

In [1]:
!pip install --upgrade py-feat


In [2]:


"""
🎬 Micro-Expression Video Preprocessing with AU Extraction (Colab Optimized)
────────────────────────────────────────────────────────────────────────────
- Maps CASME2 emotions to 6 target categories
- Extracts faces from each frame using OpenCV Haar Cascade
- Saves frames into emotion-based directory structure
- Batch extracts Facial Action Units (AUs) using py-feat for all frames in a video
- Saves per-video AU sequences as .npy for training fusion1 models
"""

import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from feat import Detector

In [3]:
# === Paths ===
BASE_DIR = '/content/drive/MyDrive/deepfake-detection'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
VIDEO_DIR = os.path.join(DATASET_DIR, 'CASME2')
LABEL_FILE = os.path.join(DATASET_DIR, 'CASME2-coding-20140508.xlsx')
OUTPUT_DIR = os.path.join(DATASET_DIR, 'microexpression_processed')

# === Ensure output directory exists ===
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Load CASME2 label file ===
df = pd.read_excel(LABEL_FILE)
df = df.dropna(subset=['Estimated Emotion'])
df['Filename'] = df['Filename'].str.strip()
df['Estimated Emotion'] = df['Estimated Emotion'].str.lower().str.strip()

In [4]:

# === Map emotions to 6 categories ===
def map_emotion(raw):
    raw = raw.lower()
    if raw == 'happiness':
        return 'happiness'
    elif raw == 'disgust':
        return 'disgust'
    elif raw == 'repression':
        return 'repression'
    elif raw == 'surprise':
        return 'surprise'
    elif raw == 'sadness':
        return 'sadness'
    elif raw in ['fear', 'others', 'anger', 'contempt']:
        return 'others'
    else:
        return None

df['Emotion'] = df['Estimated Emotion'].apply(map_emotion)
df = df.dropna(subset=['Emotion'])

In [5]:
# === Create emotion folders ===
emotion_classes = ['happiness', 'disgust', 'repression', 'surprise', 'sadness', 'others']
for ec in emotion_classes:
    os.makedirs(os.path.join(OUTPUT_DIR, ec), exist_ok=True)

In [7]:
# === Initialize face detector ===
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# === Initialize AU extractor (faster models for Colab) ===
detector = Detector(face_model="mtcnn", landmark_model="mobilefacenet", au_model="svm")

def extract_face(frame):
    """Extract and resize face from frame."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    if len(faces):
        x, y, w, h = faces[0]
        x, y = max(x, 0), max(y, 0)
        face = frame[y:y+h, x:x+w]
        return cv2.resize(face, (224, 224))
    return None

100%|██████████| 1.56M/1.56M [00:00<00:00, 24.0MB/s]
100%|██████████| 28.6k/28.6k [00:00<00:00, 9.88MB/s]
100%|██████████| 403k/403k [00:00<00:00, 9.50MB/s]
100%|██████████| 214k/214k [00:00<00:00, 6.19MB/s]
100%|██████████| 33.6M/33.6M [00:00<00:00, 131MB/s]
100%|██████████| 130k/130k [00:00<00:00, 5.35MB/s]
100%|██████████| 45.9M/45.9M [00:00<00:00, 158MB/s]
100%|██████████| 130k/130k [00:00<00:00, 5.06MB/s]
100%|██████████| 53.9M/53.9M [00:01<00:00, 33.2MB/s]
100%|██████████| 130k/130k [00:00<00:00, 5.12MB/s]
100%|██████████| 944/944 [00:00<00:00, 1.63MB/s]
100%|██████████| 170M/170M [00:01<00:00, 123MB/s]
100%|██████████| 176/176 [00:00<00:00, 208kB/s]
100%|██████████| 112M/112M [00:00<00:00, 150MB/s]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 110MB/s]


In [18]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from feat import Detector

# === Paths ===
VIDEO_DIR = "/content/drive/MyDrive/deepfake-detection/dataset/CASME2"
META_FILE = "/content/drive/MyDrive/deepfake-detection/dataset/CASME2-coding-20140508.xlsx"
OUTPUT_DIR = "/content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Emotion mapping ===
def map_emotion(raw):
    raw = raw.strip().lower()
    if raw == 'happiness':
        return 'happiness'
    elif raw == 'disgust':
        return 'disgust'
    elif raw == 'repression':
        return 'repression'
    elif raw == 'surprise':
        return 'surprise'
    elif raw == 'sadness':
        return 'sadness'
    elif raw in ['fear', 'others', 'anger', 'contempt']:
        return 'others'
    else:
        return 'others'

# === Load metadata ===
df = pd.read_excel(META_FILE)
df['Emotion'] = df['Estimated Emotion'].apply(map_emotion)

# === Initialize detector ===
detector = Detector(face_model="mtcnn", landmark_model="mobilefacenet", au_model="svm")

# === Face extraction ===
def extract_face(frame):
    try:
        detections = detector.detect_faces(frame)
        if detections is None or len(detections) == 0:
            return None

        if hasattr(detections, "iterrows"):
            first_row = next(detections.iterrows())[1]
            x1 = int(first_row['x'])
            y1 = int(first_row['y'])
            x2 = x1 + int(first_row['width'])
            y2 = y1 + int(first_row['height'])
        else:
            return None

        return frame[y1:y2, x1:x2]
    except Exception as e:
        print(f"⚠️ Face extraction failed: {e}")
        return None

# === Direct AU extraction helper ===
def extract_aus_from_face(face_img):
    try:
        # py-feat expects file path, so save temp image
        temp_path = "temp_face.jpg"
        cv2.imwrite(temp_path, face_img)
        feats = detector.detect_image([temp_path], output_size=(224, 224), batch_size=1)
        aus_tensor = feats.aus()
        if hasattr(aus_tensor, "to_numpy"):
            return aus_tensor.to_numpy().flatten()
        else:
            return aus_tensor.detach().numpy().flatten()
    except Exception as e:
        print(f"⚠️ AU extraction error: {e}")
        return np.zeros(12)

# === Process videos ===
for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing CASME2 videos"):
    video_name = row['Filename']
    emotion = row['Emotion']
    subject = row['Subject']

    video_file = video_name if video_name.endswith('.avi') else video_name + '.avi'
    video_path = os.path.join(VIDEO_DIR, f"sub{int(subject):02d}", video_file)

    if not os.path.exists(video_path):
        print(f"⚠️ Missing video: {video_path}")
        continue

    save_dir = os.path.join(OUTPUT_DIR, emotion, video_name.split('.')[0])
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    frame_idx = 0
    aus_list = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        face = extract_face(frame)
        if face is not None:
            face = cv2.resize(face, (224, 224))
            frame_path = os.path.join(save_dir, f"{frame_idx:04d}.jpg")
            cv2.imwrite(frame_path, face)

            aus = extract_aus_from_face(face)
            aus_list.append(aus)

        frame_idx += 1

    cap.release()

    if aus_list:
        np.save(os.path.join(save_dir, "AUs.npy"), np.array(aus_list))
    else:
        np.save(os.path.join(save_dir, "AUs.npy"), np.zeros((1, 12)))

print("✅ Preprocessing complete.")
print(f"📁 Output saved to: {OUTPUT_DIR}")


Processing CASME2 videos:  82%|████████▏ | 210/255 [27:14<09:19, 12.43s/it]

⚠️ Missing video: /content/drive/MyDrive/deepfake-detection/dataset/CASME2/sub23/EP03_14f.avi


Processing CASME2 videos: 100%|██████████| 255/255 [34:34<00:00,  8.13s/it]

✅ Preprocessing complete.
📁 Output saved to: /content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed
